# Smart Agriculture Weather Prediction using Machine Learning

This notebook recommends the best crop based on weather and soil conditions using a trained ML pipeline.

**Improvements over the original version:**
- Larger, realistic synthetic dataset
- Feature scaling and stratified train/test split
- Multiple model comparison with cross-validation
- Classification metrics and confusion matrix
- Confidence scores for predictions
- Structured analysis and visualizations


In [ ]:
# Import Libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
RANDOM_STATE = 42
TEST_SIZE = 0.2
FEATURES = ["Temperature", "Humidity", "Soil_Moisture", "Water_Level"]
TARGET = "Crop"


In [ ]:
# ==========================================
# DATASET (expanded from original sample)
# ==========================================
data = {
    "Temperature": [30, 25, 35, 28, 32, 26, 29, 31, 33, 27, 24, 36, 22, 34, 30,
                    28, 31, 26, 35, 29, 27, 33, 25, 32, 30, 28, 34, 26, 31, 29,
                    24, 35, 27, 32, 30, 28, 33, 25, 31, 29],
    "Humidity": [75, 60, 40, 80, 70, 55, 65, 85, 45, 72, 58, 38, 82, 42, 68,
                 77, 63, 52, 41, 74, 61, 44, 78, 69, 66, 73, 39, 56, 71, 64,
                 59, 37, 76, 67, 62, 79, 43, 57, 70, 65],
    "Soil_Moisture": [60, 45, 30, 70, 65, 40, 50, 75, 28, 68, 42, 25, 72, 32, 58,
                      62, 48, 38, 29, 66, 44, 31, 69, 63, 55, 61, 27, 41, 64, 52,
                      43, 26, 71, 60, 54, 73, 30, 39, 67, 51],
    "Water_Level": [120, 80, 20, 150, 110, 60, 90, 140, 18, 125, 70, 15, 145, 22, 100,
                    115, 85, 55, 19, 130, 75, 21, 135, 105, 95, 118, 17, 58, 108, 88,
                    72, 16, 138, 102, 92, 142, 20, 62, 112, 86],
    "Crop": ["Rice", "Wheat", "Millets", "Rice", "Sugarcane", "Wheat", "Maize", "Rice",
             "Millets", "Rice", "Wheat", "Millets", "Rice", "Millets", "Maize",
             "Rice", "Maize", "Wheat", "Millets", "Rice", "Wheat", "Millets", "Rice",
             "Sugarcane", "Maize", "Rice", "Millets", "Wheat", "Sugarcane", "Maize",
             "Wheat", "Millets", "Rice", "Sugarcane", "Maize", "Rice", "Millets",
             "Wheat", "Sugarcane", "Maize"]
}

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"
Crop distribution:
{df['Crop'].value_counts()}")
df.head(10)


In [ ]:
# ==========================================
# EXPLORATORY DATA ANALYSIS
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("Feature Distributions by Crop", fontsize=14, fontweight="bold")

for ax, feature in zip(axes.flat, FEATURES):
    sns.boxplot(data=df, x="Crop", y=feature, ax=ax, palette="Set2")
    ax.tick_params(axis="x", rotation=30)
    ax.set_title(feature)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df[FEATURES].corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# PREPARE DATA
# ==========================================
X = df[FEATURES]
y = df[TARGET]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")


In [ ]:
# ==========================================
# MODEL COMPARISON
# ==========================================
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, max_depth=6),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
}

results = []
best_model = None
best_score = -1

for name, clf in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", clf)
    ])
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="accuracy")
    pipeline.fit(X_train, y_train)
    test_pred = pipeline.predict(X_test)
    test_acc = accuracy_score(y_test, test_pred)

    results.append({
        "Model": name,
        "CV Mean Accuracy": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Test Accuracy": test_acc
    })

    if cv_scores.mean() > best_score:
        best_score = cv_scores.mean()
        best_model = pipeline
        best_model_name = name

results_df = pd.DataFrame(results).sort_values("CV Mean Accuracy", ascending=False)
print("Model Comparison Results:")
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"
Best Model: {best_model_name}")


In [ ]:
# ==========================================
# EVALUATE BEST MODEL
# ==========================================
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("
Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# PREDICTION & ANALYSIS FUNCTIONS
# ==========================================

def analyze_soil_condition(soil_moisture):
    if soil_moisture < 40:
        return "Dry Soil", "Irrigation Required"
    if soil_moisture < 70:
        return "Moderate Moisture", "Normal Irrigation"
    return "Wet Soil", "No Irrigation Required"


def analyze_rainfall(humidity, water_level):
    if humidity > 80 and water_level > 130:
        return "High Chance of Rainfall"
    return "Normal Weather Conditions"


def analyze_temperature(temperature):
    if temperature > 35:
        return "High Temperature"
    if temperature < 20:
        return "Low Temperature"
    return "Moderate Temperature"


def predict_crop(temperature, humidity, soil_moisture, water_level, model=best_model):
    sample = pd.DataFrame([[temperature, humidity, soil_moisture, water_level]], columns=FEATURES)
    prediction_idx = model.predict(sample)[0]
    probabilities = model.predict_proba(sample)[0]
    crop = label_encoder.inverse_transform([prediction_idx])[0]
    confidence = probabilities[prediction_idx] * 100

    soil_status, irrigation = analyze_soil_condition(soil_moisture)
    rain_alert = analyze_rainfall(humidity, water_level)
    temp_status = analyze_temperature(temperature)

    print("=" * 45)
    print("       SMART AGRICULTURE PREDICTION")
    print("=" * 45)
    print(f"Recommended Crop : {crop}")
    print(f"Confidence       : {confidence:.1f}%")
    print("-" * 45)
    print(f"Soil Condition   : {soil_status}")
    print(f"Irrigation       : {irrigation}")
    print(f"Rain Alert       : {rain_alert}")
    print(f"Temperature      : {temp_status}")
    print("=" * 45)

    prob_df = pd.DataFrame({
        "Crop": label_encoder.classes_,
        "Probability (%)": probabilities * 100
    }).sort_values("Probability (%)", ascending=False)
    print("
All Crop Probabilities:")
    print(prob_df.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

    return crop, confidence


In [ ]:
# ==========================================
# EXAMPLE PREDICTION (using sample values)
# ==========================================
# Original sample: Temperature=30, Humidity=75, Soil_Moisture=60, Water_Level=120
predict_crop(temperature=30, humidity=75, soil_moisture=60, water_level=120)


In [ ]:
# ==========================================
# INTERACTIVE PREDICTION (uncomment to use)
# ==========================================
# print("\n====== ENTER WEATHER DETAILS ======\n")
# temperature = float(input("Enter Temperature: "))
# humidity = float(input("Enter Humidity: "))
# soil_moisture = float(input("Enter Soil Moisture: "))
# water_level = float(input("Enter Water Level: "))
# predict_crop(temperature, humidity, soil_moisture, water_level)


In [ ]:
# ==========================================
# COMPREHENSIVE VISUALIZATION DASHBOARD
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Smart Agriculture Weather Analysis Dashboard", fontsize=14, fontweight="bold")

axes[0, 0].plot(df.index, df["Temperature"], marker="o", color="#e74c3c", linewidth=2)
axes[0, 0].set_title("Temperature Trend")
axes[0, 0].set_xlabel("Records")
axes[0, 0].set_ylabel("°C")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].bar(df.index, df["Humidity"], color="#3498db", alpha=0.8)
axes[0, 1].set_title("Humidity Levels")
axes[0, 1].set_xlabel("Records")
axes[0, 1].set_ylabel("%")
axes[0, 1].grid(True, alpha=0.3, axis="y")

scatter = axes[1, 0].scatter(df["Temperature"], df["Soil_Moisture"],
                             c=df["Humidity"], cmap="viridis", s=80, edgecolors="white")
axes[1, 0].set_title("Temperature vs Soil Moisture (colored by Humidity)")
axes[1, 0].set_xlabel("Temperature (°C)")
axes[1, 0].set_ylabel("Soil Moisture (%)")
plt.colorbar(scatter, ax=axes[1, 0], label="Humidity (%)")

axes[1, 1].plot(df.index, df["Water_Level"], marker="s", color="#2ecc71", linewidth=2)
axes[1, 1].set_title("Water Level Trend")
axes[1, 1].set_xlabel("Records")
axes[1, 1].set_ylabel("Level")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSmart Agriculture Weather Prediction Completed Successfully")
